<a href="https://colab.research.google.com/github/Richard9235/NLP-Fundamentals-and-Application-Sentiment-Topic-and-Customer-Segmentation-./blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Natrual Language Processing (NLP)

<h3>Import necessary libraries</h3>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
import zipfile

<h3>Extracting the dataset</h3>

In [ ]:
zip_file_path = r"/content/drive/MyDrive/Python Bootcamp Uplift Datasets/Women's E-Commerce Clothing Reviews.zip"
file_folder_path = r"/content/drive/MyDrive/Python Bootcamp Uplift Datasets/"
with zipfile.ZipFile(zip_file_path) as zip_ref:
    zip_ref.extractall(file_folder_path)
file_path = r"/content/drive/MyDrive/Python Bootcamp Uplift Datasets/Womens Clothing E-Commerce Reviews.csv"

<h3>Initial look at the dataset</h3>

In [ ]:
df = pd.read_csv(file_path)

In [ ]:
df.head()

In [ ]:
df.tail()

<h4>Why Encoding doesn't work on raw human language</h4>

- `Traditional Encoding` only works for categorical dataset
- `Encoding` is replacing text with a number
- `One Hot Encoding` gurantees no ordinal values in the dataset

<h3>What we'll be using instead of Encoding</h3>

<h4>Vectorization</h4>

- Concept:
  - We show the model a dictionary of raw human written texts
  - `Vector Map`: Model creates a map that groups almost similar meaning and those words usually seen together
  - `Vector Map` is a guide for the model to understand human language

- The model in question are `transformer models`, specifically, `sentiment models`


<h3>Text Analysis</h3>


In [ ]:
sentiment_model = "cardiffnlp/twitter-roberta-base-sentiment" # Model Name
classifier = pipeline("sentiment-analysis", model = sentiment_model, tokenizer = sentiment_model )# Model Instance

<h5> Classifier Parameters</h5>

1. What you want the model to do (sentiment-analysis)
2. What model you want to use (model = sentiment model variable)
3. Which subset model do you want to be responsible for transforming text information into token(tokenizer = sentiment model variable)

- Tokens are used to make the vector maps
- Tokenizers look at the dataset(text) and transforms it into a map compatible format(Vector Map)
- `cardiffnlp` already comes with a prebuilt tokenizer in its model so we don't need a separate tokenizer model


In [ ]:
sample_text = "I hate you."

In [ ]:
classifier(sample_text)

The cardiffnlp model has `3` Labels:
- "Positive" (Label 2)
- "Negative" (Label 0)
- "Neutral" (Label 1)

The score is the confidence level of the model || how close the label is to the text

In [ ]:
sample_text2 = "I have work tomorrow."
classifier(sample_text2)

In [ ]:
sample_text3 = "I'm totally fine'."
classifier(sample_text3)

In [ ]:
sample_text4 = "My wedding is tomorrow."
classifier(sample_text4)

In [ ]:
sample_text5 = "Pagod na ako."
classifier(sample_text5)
# Make sure that the model used is trained using the same language as the input as it would not recognize the text sentiment

<h3> Rudimentary Cleaning in order to enforce that columns are in text format</h3>

In [ ]:
df['Review Text'] = df['Review Text'].fillna('').astype(str)

- `fillna` Takes all the NA(Not Applicable) fields to use an empty string ('') instead of NA
- `astype` converts fields into a string for users who type in numerical values

In [ ]:
df_smaller = df[:30]

In [ ]:
results = classifier(df_smaller['Review Text'].tolist())

In [ ]:
df_smaller['sentiment_label'] = [indiv_result["label"] for indiv_result in results]
df_smaller['sentiment_score'] = [indiv_score["score"] for indiv_score in results]

In [ ]:
df_smaller[['Review Text', "sentiment_label", 'sentiment_score']].head(4)

In [ ]:
df_smaller[['Review Text', "sentiment_label", 'sentiment_score']].tail(4)

Additional Information:
- `Finetunning` is customizing a pretrained model to your use case
- `Finetunning` is giving the model a list of new words with indication of positive, negative, or neutral connotations

## To Do Task:

- Run the classification model for 2000 fields
- Perform Exploratory Data analysis for the data together with the new columns(Label and Score)
- Answer the ff:
  - Identify, using graphs, identify the products with positive and negative sentiments
  - Do the same for the division, department, and class, identify the division with positive and negative sentiments
  - Inspect correlation between the categories and the sentiment score.

In [ ]:
df_task = df[:2000]

In [ ]:
df_task

In [ ]:
results = classifier(df_task['Review Text'].tolist())

In [ ]:
df_task['sentiment_label'] = [indiv_result["label"] for indiv_result in results]
df_task['sentiment_score'] = [indiv_score["score"] for indiv_score in results]

<H5> I'll be using (bar chart, boxplot, pie chart) for visualization</H5>

In [ ]:
df_task

In [ ]:
# Bar Graph
plt.figure(figsize=(10, 8))
sns.countplot(data=df_task, x='sentiment_label')

plt.title('Distribution of Review Text by Sentiment Labels ', fontsize=14, fontweight='bold')
plt.xlabel('Sentiment Labels', fontsize=12)
plt.ylabel('Count of Reviews', fontsize=12)
plt.show()

In [ ]:
#Pie Chart
labels = df_task["sentiment_label"].value_counts().index
sizes = df_task["sentiment_label"].value_counts().values

fig, ax = plt.subplots()
ax.pie(sizes, labels=labels)
plt.title('Distribution of Sentiment Labels', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
division_results = classifier(df_task['Division Name'].tolist())

In [ ]:
class_results = classifier(df_task['Class Name'].tolist())

In [ ]:
department_results = classifier(df_task['Department Name'].tolist())

## NLP Methodologies

### Dimensionality Reduction
- For datasets with more than 10 columns
- Tries to collapse columns into smaller number of columns via `multi-colinear dependency`
- `Multi-Colinear Dependency` combines columns that can be inferred from one another
- An advantage of dimensionality reduction is that it helps with `computational efficiency`

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [ ]:
num_cols = df_smaller.select_dtypes(["int","float"]).columns
cat_cols = df_smaller.select_dtypes(["object"]).columns

In [ ]:
transformation_pipeline = ColumnTransformer([
    ("num", SimpleImputer(strategy = "median"), num_cols),
    ("scaler", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(), cat_cols)
])

*Refresher: SimpleImputer just replaces the value with the median/mode/mean*

In [ ]:
df_smaller_trans = transformation_pipeline.fit_transform(df_smaller)

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca_model = PCA(n_components=2)

- `n_components` defines the amount of columns you want to reduce your dataset to
- In our case, we'd be reducing our dataset to only have 2 columns
- To identify which column number should we reduce the dataset to, it's a `trial-and-error` process to be validated by our tests.

In [ ]:
df_pca = pca_model.fit_transform(df_smaller_trans)

In [ ]:
plt.figure(figsize=(5,7))
sns.scatterplot(x=df_pca[:,0], y=df_pca[:,1])
plt.show()

### KMeans Clustering
- 1 Specific method of `segmentation/clustering`



In [ ]:
from sklearn.cluster import KMeans

In [ ]:
kmeans_model = KMeans(n_clusters=3, random_state = 42, n_init = 10)

- `n_clusters` a distinct guess of the groups of your dataset
- `random_state` puts a tag on the starting point so that it gives out consistent results when rerunning
- `n_init` how many times the model runs before the operation stops, better outcomes the higher the number with the cost of longer processing time

In [ ]:
clusters = kmeans_model.fit_predict(df_pca)

In [ ]:
plt.figure(figsize=(5,7))
sns.scatterplot(x=df_pca[:,0], y=df_pca[:,1], c=clusters)
plt.show()

In [ ]:
df_smaller['cluster'] = clusters

In [ ]:
df_smaller.head(10)

### Elbow Method

In [ ]:
inertias =  []
number_clusters = range(1,10)

for number_cluster in number_clusters:
  kmeans_model = KMeans(n_clusters=number_cluster, random_state = 42, n_init = 10)
  kmeans_model.fit(df_pca)
  inertias.append(kmeans_model.inertia_)

In [ ]:
plt.figure(figsize=(10,4))
sns.scatterplot(x=number_clusters, y= inertias)
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia Score")
plt.title("Number of Clusters vs Inertia Score")
plt.show()

- `Inertia` is the likelyhood of the data defecting from the cluster
- Elbow Method identifies the optimal number of clusters to use for KMeans.
- In our case, the elbow is located at `3`, therefore, we should use 3 for our `n-clusters` at our KMeans model instance
- Should be performed before implementing the KMeans
- The lower the inertia score, the better

<h6>Review of Process</h6>

1. Perform Dimensionality Reduction to reduce number of columns to `2`
2. Plot a 2 dimensional graph
3. Perform the elbow method to find the optimal `n_clustering` number
4. Perform KMeans clustering, inputting `n_clustering` with the optimal number.
5. Create a graph again for KMeans but this time to specify differnt colors for different clusters
6. Lastly, we create a new column on our dataset with our identified clusters

<h5>Interview Review</h5>

1. How would you explain Sentiment Analysis, Dimensionality Reduction, and Customer Segmentation?

Answer:
- Sentiment Analysis deals with the combination of vectorization and classifiers wherein classifiers are the models themselves acting as a scorer to identify whether the input's sentiment is positive, negative, or neutral whilst vectorization is the act of transforming our data into vector maps or as Sir Cielo described it, geolocation coordinates for the model to use.
- Dimensionality Reduction is the act of abstracting our dataset by reducing the number of our columns to our desired number, in our case we reduced our fashion review dataset to 2
- Customer Segmentation is the combination of both KMeans clustering and the elbow method wherein we identified the optimal number of clusters via the elbow method and used that to group our datas into clusters via KMeans. We then created a new column to visualize the results of KMeans.

2. Explain it in one sentence

Answer:
- Sentiment Analysis is the act of identifying the sentiment of the user input, Dimensionality Reduction is reducing the amount of columns of a given dataset, and Customer Segmentation is the act of clustering fields in our dataset

### Topic Classification

In [ ]:
from transformers import pipeline

In [ ]:
candidate_topics = ['Fast Delivery', ' Good Communication', 'Quality of Products', 'Handling', 'Price', 'Packaging']

*We're always required to create a list of candidate topics*

In [ ]:
topic_classifier = pipeline("zero-shot-classification", model = 'facebook/bart-large-mnli')

- Zero Shot is named zero shot because we don't have to train the model in order for it to classify topics
- Regular classification requires us to teach the model the dataset inorder for it to classify data, whilst zero shot doesn't require dataset training anymore because it comes with it out of the box.

In [ ]:
sample_review = "I received the product on time."

In [ ]:
result = topic_classifier(sample_review, candidate_topics)

In [ ]:
result

In [ ]:
result['labels'][0], result['scores'][0]*100

In [ ]:
sample_review2 = "The jeans are decent for the price point but the shipping took nearly two weeks to arrive and the packaging was slightly torn when it got here."
result2 = topic_classifier(sample_review2, candidate_topics)
result2

In [ ]:
df_smaller.head()

In [ ]:
customer_reviews = df_smaller["Review Text"].tolist()

In [ ]:
topics = []
for review in customer_reviews:
  result = topic_classifier(review, candidate_topics)
  topic = result["labels"][0]
  topics.append(topic)


In [ ]:
df_smaller['topics'] = topics
df_smaller[['Review Text','topics']]

In [ ]:
df_smaller['topics'].value_counts()

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
filt = df_smaller["topics"] == "Packaging"
df_smaller[filt]


## To Do
1. Find your own dataset (Look for one on Kaggle)
2. Apply any of the four NLP methodologies
3. Visualization is applied twice, once on the raw dataset and once after the methodology

### Extracting the Dataset

In [ ]:
zip_file_path = r"/content/drive/MyDrive/Python Bootcamp Uplift Datasets/Facebook Reviews.zip"
file_folder_path = r"/content/drive/MyDrive/Python Bootcamp Uplift Datasets/"
with zipfile.ZipFile(zip_file_path) as zip_ref:
    zip_ref.extractall(file_folder_path)
fb = r"/content/drive/MyDrive/Python Bootcamp Uplift Datasets/facebook_reviews.csv"

### Initial Look at the data

In [ ]:
df_fb = pd.read_csv(fb)
df_fb.head()


In [ ]:
df_fb.tail()

In [ ]:
rows, columns = df.shape
print('Number of Rows:', rows)
print('Number of Columns:', columns)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

### Data Visualization

### Initial Insights

### Initial Feature Engineering

In [ ]:
df_fb.dropna(inplace=True)
df_fb.isnull().sum()

### Exploratory Analysis

### Sentiment Analysis

In [ ]:
sentiment_model = "cardiffnlp/twitter-roberta-base-sentiment" # Model Name
classifier = pipeline("sentiment-analysis", model = sentiment_model, tokenizer = sentiment_model )# Model Instance

In [ ]:
df_fb_smaller = df_fb[:2000]

In [ ]:
results = classifier(df_fb_smaller['content'].tolist(), truncation=True)
df_fb_smaller['sentiment_label'] = [indiv_result["label"] for indiv_result in results]
df_fb_smaller['sentiment_score'] = [indiv_score["score"] for indiv_score in results]

In [ ]:
df_fb_smaller

In [ ]:
# Bar Graph
plt.figure(figsize=(10, 8))
sns.countplot(data=df_fb_smaller, x='sentiment_label')

plt.title('Distribution of Review Text by Sentiment Labels ', fontsize=14, fontweight='bold')
plt.xlabel('Sentiment Labels', fontsize=12)
plt.ylabel('Count of Reviews', fontsize=12)
plt.show()

### Data Preprocessing

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


In [ ]:
df_fb.head(5)

In [ ]:

df_fb_smaller['content'] = df_fb_smaller['content'].fillna('').astype(str)

num_cols = df_fb_smaller.select_dtypes(["int","float"]).columns

cat_cols = df_fb_smaller.select_dtypes(["object"]).columns

transformation_pipeline = ColumnTransformer([
    ("num", SimpleImputer(strategy = "median"), num_cols),
    ("scaler", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(), cat_cols)
])

df_fb_smaller_trans = transformation_pipeline.fit_transform(df_fb_smaller)

### Dimensionality Reduction

In [ ]:
df_fb_smaller = transformation_pipeline.fit_transform(df_fb_smaller)

In [ ]:
from sklearn.decomposition import PCA
pca_model = PCA(n_components=2)

In [ ]:
df_pca_fb = pca_model.fit_transform(df_fb_smaller)

In [ ]:
plt.figure(figsize=(5,7))
sns.scatterplot(x=df_pca_fb[:,0], y=df_pca_fb[:,1])
plt.show()

### Elbow Method

In [ ]:
inertias =  []
number_clusters = range(1,10)

for number_cluster in number_clusters:
  kmeans_model = KMeans(n_clusters=number_cluster, random_state = 42, n_init = 10)
  kmeans_model.fit(df_pca_fb)
  inertias.append(kmeans_model.inertia_)

In [ ]:
plt.figure(figsize=(10,4))
sns.scatterplot(x=number_clusters, y= inertias)
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia Score")
plt.title("Number of Clusters vs Inertia Score")
plt.show()

### KMeans

In [ ]:
from sklearn.cluster import KMeans
kmeans_model = KMeans(n_clusters=3, random_state = 42, n_init = 10)
clusters = kmeans_model.fit_predict(df_pca_fb)

In [ ]:
plt.figure(figsize=(5,7))
sns.scatterplot(x=df_pca_fb[:,0], y=df_pca_fb[:,1], c=clusters)
plt.show()

### Topic Classification

In [ ]:
from transformers import pipeline
from tqdm.notebook import tqdm

In [ ]:
topic_classifier = pipeline("zero-shot-classification", model = 'facebook/bart-large-mnli')
df_fb_smaller = df_fb[:30] # Re-initialize df_fb_smaller to be a DataFrame
customer_reviews = df_fb_smaller["content"].tolist()
candidate_topics = [
    "sentiment_analysis",
    "spam_detection",
    "fake_review_detection",
    "rating_prediction",
    "topic_modeling",
    "emotion_detection",
    "bug_detection",
    "review_summarization",
    "keyword_extraction",
    "review_classification",
    "toxicity_detection",
    "review_clustering",
    "trend_analysis",
    "recommendation_system",
    "user_satisfaction_analysis",
    "app_feature_analysis",
    "complaint_detection",
    "intent_detection",
    "language_detection",
    "sarcasm_detection"
]
topics = []
for review in tqdm(customer_reviews, desc="Classifying topics"):
  result = topic_classifier(review, candidate_topics)
  topic = result["labels"][0]
  topics.append(topic)

In [ ]:
df_fb_smaller['topics'] = topics
df_fb_smaller[['content','topics']]


In [ ]:
df_fb_smaller['topics'].value_counts()

### Exploratory Analysis

### Conclusion

### LLM Integration

- Sir said that we cannot be replaced by AI because there is no accountability in their findings.

- I'll use this to import an LLM to explain why they laid off a person or why they came to a conclusion

- The way I'd use an LLM for the project is to gain a reasoning model